In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
# %%
import datetime
import logging
import os

import pandas as pd
# /venv/lib/python3.12/site-packages/gspread_pandas/spread.py:401: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)` .replace("", np.nan)
pd.set_option('future.no_silent_downcasting', True)

import helpers.hdbg as hdbg
import helpers.henv as henv
import helpers.hpandas as hpandas
import helpers.hprint as hprint
import helpers.hcache as hcache

#hcache.get_global_cache_info()
#hcache.clear_global_cache("all")

import config_root.config as cconfig

# %%
hdbg.init_logger(verbosity=logging.INFO)

_LOG = logging.getLogger(__name__)

_LOG.info("%s", henv.get_system_signature()[0])

hprint.config_notebook()

INFO  > cmd='/venv/lib/python3.12/site-packages/ipykernel_launcher.py -f /home/.local/share/jupyter/runtime/kernel-4d7a690f-3207-4cbe-ac0b-80878bb13a1c.json'
INFO  # Git
  branch_name='CmampTask11020_Compute_yamm_stats'
  hash='aa9e3a7dc'
  # Last commits:
    * aa9e3a7dc GP Saggese Update                                                            (  11 hours ago) Sun Jan 5 03:47:04 2025  (HEAD -> CmampTask11020_Compute_yamm_stats, origin/CmampTask11020_Compute_yamm_stats)
    * a3b8889e8 GP Saggese Update                                                            (  12 hours ago) Sun Jan 5 02:27:58 2025           
    * 79bc7e066 GP Saggese Update                                                            (  22 hours ago) Sat Jan 4 16:13:31 2025           
# Machine info
  system=Linux
  node name=ce6cfb94b060
  release=6.6.22-linuxkit
  version=#1 SMP Fri Mar 29 12:21:27 UTC 2024
  machine=aarch64
  processor=aarch64
  cpu count=8
  cpu freq=None
  memory=svmem(total=8222072832, avai

In [11]:
#!rm -rf /mnt/*

In [13]:
import gspread
print(gspread.__version__)

import gspread_pandas
print(gspread_pandas.__version__)

#gspread_pandas.conf.get_config()
print(gspread_pandas.conf.get_config()["project_id"])

#!sudo /bin/f bash -c "(source /venv/bin/activate; pip install --upgrade google-api-python-client)"

import importlib
import ck_marketing.process_automation.hyamm as hyamm
importlib.reload(hyamm)

#import ck_marketing.hunterio.hunter_api as cmhuhuap
#importlib.reload(cmhuhuap)

import helpers.hopenai as hopenai

5.12.4
3.3.0
gspread-gp


# Read data

In [151]:
url = "https://docs.google.com/spreadsheets/d/1p4oqnbYKn1voXIDilyHNdqoM_sICM-jgSqZ14AiDGLs"
contact_df_enriched = hyamm.get_cached_sheet_to_df(url, "Sheet1")
print(contact_df_enriched.shape)
display(contact_df_enriched.head(2))

INFO  Loading cached version from memory ...
INFO  Loading cached version from memory done (0.080 s)
(4000, 24)


,hash,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain
0,00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",,,,,,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com
1,0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",,,,,,,FALSE,TRUE,FALSE,0,


In [152]:
print("shape=", contact_df_enriched.shape)
contact_df_enriched["origin"].value_counts()

shape= (4000, 24)


origin
GP_LIn_connections          1212
Folkapp                     1112
VCSheet_Query1               479
Search4.FinTech_VC_in_US     468
Euro-VC-LinkedIn             298
VC_search_export             159
VC Tier 2                    143
VC Tier 1                    129
Name: count, dtype: int64

In [153]:
contact_df_enriched.set_index("hash", inplace=True)

## Sanity check

In [154]:
# Remove useless data.
#cols = "origin first_name last_name category company_domain company_name job_title stages".split()
#contact_df_enriched = contact_df_enriched[cols]

In [168]:
hyamm.get_value_counts_stats_df(contact_df_enriched, "category")

,count,pct [%]
category,,
,1804,45.100
Venture Fund,1004,25.100
Financial Services,274,6.850
Venture Capital & Private Equity,176,4.400
Computer Software,163,4.075
Higher Education,133,3.325
Information Technology & Services,73,1.825
Accelerator,45,1.125
Corporate VC,32,0.800


In [169]:
contact_df_tmp = hyamm.infer_category(contact_df_enriched)

,num,pct [%]
tag,,
stages,1407,35.175
partner_in_job,1330,33.250
venture_in_name,886,22.150
capital_in_company_name,673,16.825
vc_in_domain,482,12.050
venture_in_job,290,7.250
director_in_job,269,6.725
invest_in_job,230,5.750
vc_in_name,78,1.950


,count,pct [%]
is_vc,,
0,2448,61.200
1,635,15.875
2,543,13.575
3,279,6.975
4,90,2.250
5,5,0.125


In [170]:
hyamm.get_value_counts_stats_df(contact_df_tmp, "category")

,count,pct [%]
category,,
Venture Fund (inferred),1552,38.800
Venture Fund,1004,25.100
Financial Services,274,6.850
,252,6.300
Venture Capital & Private Equity,176,4.400
Computer Software,163,4.075
Higher Education,133,3.325
Information Technology & Services,73,1.825
Accelerator,45,1.125


In [171]:
hpandas.filter_df(contact_df_tmp, "category", "Venture Fund (inferred)").head(5)

INFO  selected=1552 / 4000 = 38.80%


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain
hash,,,,,,,,,,,,,,,,,,,,,,,
00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",False,,,Venture Fund (inferred),,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com
000fcbecf9553942a40ec2ad0f8d88b3,Search4.FinTech_VC_in_US,2023-11-04T04:30:39.811Z,Luiz,Noronha,noronha@dnacapital.com,,https://www.linkedin.com/in/luizhnoronha/,Partner,Helping entrepreneurs build generational compa...,DNA Capital,dnacapital.ca,"Palo Alto, California, United States",False,,,Venture Fund (inferred),,dnacapital.com,FALSE,TRUE,FALSE,0,
0041bcd7d2bf911652720b947f7e074b,Search4.FinTech_VC_in_US,2023-11-11T14:35:44.417Z,Phil,Herget,ph@ardent.vc,,https://www.linkedin.com/in/phil-herget/,General Partner and Co-Founder,,Ardent Venture Partners,ardent.vc,Washington D.C. Metro Area,False,,,Venture Fund (inferred),,ardent.vc,FALSE,TRUE,FALSE,0,
00472f7b45ed278dbc1cde93c35fac90,VC_search_export,2024-07-11T18:23:40.118Z,Andrea,Wang,awang@generalcatalyst.com,unknown,https://www.linkedin.com/in/ACwAAAwuMv0BEz4ZcI...,Partner,"Partner to early-stage (pre-seed, seed, series...",General Catalyst,generalcatalyst.com,"San Francisco, California, United States",False,,,Venture Fund (inferred),,generalcatalyst.com,FALSE,TRUE,FALSE,0,
004b2950e50141a85c2a7b8998132709,VCSheet_Query1,,Matt,Bigge,mbigge@crosslinkcapital.com,,https://www.linkedin.com/in/bigge/,Partner@Crosslink Capital,,Crosslink Capital,crosslinkcapital.com,,False,,,Venture Fund (inferred),,crosslinkcapital.com,FALSE,TRUE,FALSE,0,


In [172]:
hpandas.filter_df(contact_df_tmp, "category", "").head()

INFO  selected=252 / 4000 = 6.30%


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain
hash,,,,,,,,,,,,,,,,,,,,,,,
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",False,,,,,,FALSE,TRUE,FALSE,0,
00b9de3a32b6af2c2dec58193e94ce9f,VCSheet_Query1,,Mahdi,Raza,mahdi@exponentcap.co,,https://www.linkedin.com/in/mahdiraza92/,Founder@Exponent,,Exponent,exponent.com,,False,,,,,exponentcap.co,FALSE,TRUE,FALSE,0,
013c3edff3b4ca24ecc4b4f616efb558,GP_LIn_connections,2024-12-31T02:21:49.506Z,Pete,Wassell,pete@carboneraexchange.com,,https://linkedin.com/in/petewassell,Chairman,"Executive in the field of Software, Technology...",Carbonera Exchange,carboneraexchange.com,New York City Metropolitan Area,False,,,,,,FALSE,TRUE,FALSE,0,
01797b1198edb0f3927a7b0aef6f6d6c,GP_LIn_connections,2024-12-31T19:15:56.917Z,Fiona,Smith Foddai,_nan_,,https://linkedin.com/in/fiona-smith-foddai-936...,PATIENT COORDINATOR FOR ASSISTED REPRODUCTION .,Fertility Specialist IVF Coordinator; ;,FIV OCHOA HOSPITAL,_nan_,Greater Marbella Metropolitan Area,False,,,,,,FALSE,TRUE,FALSE,0,
02e73ac95db050c2d95585ca3e6ba381,Search4.FinTech_VC_in_US,2023-11-12T14:34:57.731Z,Yubo,Ruan,_nan_,,https://www.linkedin.com/in/yubo-ruan/,Founder,Yubo is a two-time Stanford University dropout...,ParaSpace,para.space,San Francisco Bay Area · Remote,False,,,,,,FALSE,TRUE,FALSE,0,


## Use ChatGpt

In [ ]:
#!sudo /bin/bash -c "(source /venv/bin/activate; pip install --quiet openai)"

In [ ]:
hopenai.start_logging_costs()

prompt = """
I will give you a list of job titles on different rows in a number list.
Return True for each job title that corresponds to somebody 
making investment decisions at a venture firm.
Return False for each job title that corresponds to somebody who is not at a venture firm.
If you are not sure, just respond "nan".
Each response should go on a different row, prepended by the number of the job
titles.
"""
df["input"] = df["job_title"] + " at " + df["company_name"]

models = [
    #"gpt-3.5-turbo",
    "gpt-4o-mini",
    #"gpt-4o"
]

for model in models:
    input_col = "input"
    response_col = model
    allow_overwrite = True
    df = hopenai.apply_prompt_to_dataframe(df, prompt,
                                          model, input_col, response_col, allow_overwrite=allow_overwrite)
print("cost=", hopenai.get_costs())

In [ ]:
mask = df["gpt-4o-mini"] != "True"
print(mask.sum())

cols = ["job_title", "company_name", "gpt-4o-mini"]
df.loc[mask][cols].head()

## Categorize

In [124]:
#print("\n".join(sorted(contact_df_tmp["category"].unique())))
print(contact_df_enriched["category"].value_counts().head(10))

category
                                     1804
Venture Fund                         1004
Financial Services                    274
Venture Capital & Private Equity      176
Computer Software                     163
Higher Education                      133
Information Technology & Services      73
Accelerator                            45
Corporate VC                           32
Research                               21
Name: count, dtype: int64


In [105]:
keyword = "Venture Capital & Private Equity"
#mask = [keyword in val.lower() for val in contact_df_tmp["category"]]
mask = contact_df_enriched["category"] == keyword
print(hprint.perc(sum(mask), len(mask)))

176 / 4000 = 4.40%


In [106]:
hpandas.filter_df(contact_df_enriched, "category", "").head(2)

INFO  selected=1804 / 4000 = 45.10%


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain
hash,,,,,,,,,,,,,,,,,,,,,,,
00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",,,,,,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",,,,,,,FALSE,TRUE,FALSE,0,


In [107]:
mask = contact_df_enriched["category"] == ""
print(hprint.perc(sum(mask), len(mask)))

#print(contact_df_enriched[mask]["stages"].unique())

contact_df_tmp = contact_df_enriched[mask]

1804 / 4000 = 45.10%


In [167]:
masks = {}

def _append_mask(mask, tag):
    #tag = len(masks)
    masks[tag] = mask
    print("%s: %s" % (tag, hprint.perc(sum(mask), len(mask))))

keyword = "vc"
mask = [keyword in val.lower() for val in contact_df_tmp["company_domain"]]
_append_mask(mask, "vc_in_domain")

keyword = "vc"
mask = [keyword in val.lower() for val in contact_df_tmp["company_name"]]
_append_mask(mask, "vc_in_name")

keyword = "venture"
mask = [keyword in val.lower() for val in contact_df_tmp["company_name"]]
_append_mask(mask, "venture_in_name")

mask = contact_df_tmp["stages"] != ""
_append_mask(mask, "stages")

keyword = "partner"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "partner_in_job")

keyword = "vc"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "vc_in_job")

keyword = "invest"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "invest_in_job")

keyword = "venture"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "venture_in_job")

keyword = "director"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "director_in_job")

keyword = "scout"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "scout_in_job_title")

keyword = "eir"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "eir_in_job_title")

keyword = "in residence"
mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
_append_mask(mask, "in_residence_in_job_title")

# keyword = "angel"
# mask = [keyword in val.lower() for val in contact_df_tmp["job_title"]]
# _append_mask(mask, "angel_in_job_title")

keyword = "capital"
mask = [keyword in val.lower() for val in contact_df_tmp["company_name"]]
_append_mask(mask, "capital_in_company_name")

#display(contact_df_tmp[mask].head(10))

vc_in_domain: 482 / 4000 = 12.05%
vc_in_name: 78 / 4000 = 1.95%
venture_in_name: 886 / 4000 = 22.15%
stages: 4000 / 4000 = 100.00%
partner_in_job: 1330 / 4000 = 33.25%
vc_in_job: 31 / 4000 = 0.78%
invest_in_job: 230 / 4000 = 5.75%
venture_in_job: 290 / 4000 = 7.25%
director_in_job: 269 / 4000 = 6.73%
scout_in_job_title: 18 / 4000 = 0.45%
eir_in_job_title: 0 / 4000 = 0.00%
in_residence_in_job_title: 13 / 4000 = 0.33%
capital_in_company_name: 673 / 4000 = 16.83%


In [109]:
#contact_df_tmp[masks["vc_in_domain"]].head()
#contact_df_tmp[masks["vc_in_name"]].head(10)

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain
hash,,,,,,,,,,,,,,,,,,,,,,,
0041bcd7d2bf911652720b947f7e074b,Search4.FinTech_VC_in_US,2023-11-11T14:35:44.417Z,Phil,Herget,ph@ardent.vc,,https://www.linkedin.com/in/phil-herget/,General Partner and Co-Founder,,Ardent Venture Partners,ardent.vc,Washington D.C. Metro Area,,,,,,ardent.vc,FALSE,TRUE,FALSE,0,
00fe863a8a40caac5e4661bf3e637db1,VCSheet_Query1,,Gary,Peat,gary@valor.vc,,https://www.linkedin.com/in/garypeat/,General Partner@Valor Ventures,,Valor Ventures,valor.vc,,,,,,,valor.vc,FALSE,TRUE,FALSE,0,
0145d5935051353cb80bf10a5e6c26d7,VC Tier 2,,Gus,Domel,gus@boost.vc,valid,_nan_,Principal,,Boost VC,boost.vc,San Francisco Bay Area,,,,,,boost.vc,TRUE,TRUE,TRUE,1,boost.vc
01ac01ff7b004f16db70b707794e27a3,VCSheet_Query1,,George,Ugras,george@av8.vc,,https://www.linkedin.com/in/ugras/,General Partner@AV8 Ventures,,AV8 Ventures,av8.vc,,,,,,,av8.vc,FALSE,TRUE,FALSE,0,
01c99fdf8592d8c61a62a71ab7162a57,VCSheet_Query1,,Ryan,Holmes,ryanh@loi.vc,,https://www.linkedin.com/in/rholmes/,Co-Founder & Chairman@LOI Venture,,LOI Venture,loi.vc,,,,,,,,FALSE,TRUE,FALSE,0,


In [110]:
contact_df_tmp["strunz"] = False
#contact_df_tmp[masks["vc_in_domain"]]["strunz"] = False
contact_df_tmp.loc[masks["vc_in_domain"], "strunz"] = True
contact_df_tmp.head()

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain,strunz
hash,,,,,,,,,,,,,,,,,,,,,,,,
00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",,,,,,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com,False
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",,,,,,,FALSE,TRUE,FALSE,0,,False
000fcbecf9553942a40ec2ad0f8d88b3,Search4.FinTech_VC_in_US,2023-11-04T04:30:39.811Z,Luiz,Noronha,noronha@dnacapital.com,,https://www.linkedin.com/in/luizhnoronha/,Partner,Helping entrepreneurs build generational compa...,DNA Capital,dnacapital.ca,"Palo Alto, California, United States",,,,,,dnacapital.com,FALSE,TRUE,FALSE,0,,False
0041bcd7d2bf911652720b947f7e074b,Search4.FinTech_VC_in_US,2023-11-11T14:35:44.417Z,Phil,Herget,ph@ardent.vc,,https://www.linkedin.com/in/phil-herget/,General Partner and Co-Founder,,Ardent Venture Partners,ardent.vc,Washington D.C. Metro Area,,,,,,ardent.vc,FALSE,TRUE,FALSE,0,,True
00472f7b45ed278dbc1cde93c35fac90,VC_search_export,2024-07-11T18:23:40.118Z,Andrea,Wang,awang@generalcatalyst.com,unknown,https://www.linkedin.com/in/ACwAAAwuMv0BEz4ZcI...,Partner,"Partner to early-stage (pre-seed, seed, series...",General Catalyst,generalcatalyst.com,"San Francisco, California, United States",,,,,,generalcatalyst.com,FALSE,TRUE,FALSE,0,,False


In [111]:
for tag, mask in masks.items():
    contact_df_tmp[tag] = False
    contact_df_tmp.loc[mask, tag] = True

In [112]:
contact_df_tmp.head()

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain,strunz,vc_in_domain,vc_in_name,venture_in_name,partner_in_job,vc_in_job,invest_in_job,venture_in_job,director_in_job
hash,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",False,,,,,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com,False,False,False,False,True,False,False,False,False
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False
000fcbecf9553942a40ec2ad0f8d88b3,Search4.FinTech_VC_in_US,2023-11-04T04:30:39.811Z,Luiz,Noronha,noronha@dnacapital.com,,https://www.linkedin.com/in/luizhnoronha/,Partner,Helping entrepreneurs build generational compa...,DNA Capital,dnacapital.ca,"Palo Alto, California, United States",False,,,,,dnacapital.com,FALSE,TRUE,FALSE,0,,False,False,False,False,True,False,False,False,False
0041bcd7d2bf911652720b947f7e074b,Search4.FinTech_VC_in_US,2023-11-11T14:35:44.417Z,Phil,Herget,ph@ardent.vc,,https://www.linkedin.com/in/phil-herget/,General Partner and Co-Founder,,Ardent Venture Partners,ardent.vc,Washington D.C. Metro Area,False,,,,,ardent.vc,FALSE,TRUE,FALSE,0,,True,True,False,True,True,False,False,False,False
00472f7b45ed278dbc1cde93c35fac90,VC_search_export,2024-07-11T18:23:40.118Z,Andrea,Wang,awang@generalcatalyst.com,unknown,https://www.linkedin.com/in/ACwAAAwuMv0BEz4ZcI...,Partner,"Partner to early-stage (pre-seed, seed, series...",General Catalyst,generalcatalyst.com,"San Francisco, California, United States",False,,,,,generalcatalyst.com,FALSE,TRUE,FALSE,0,,False,False,False,False,True,False,False,False,False


In [113]:
# Make sure that there is an assignment.
contact_df_tmp[masks.keys()].sum(axis=0)

vc_in_domain        221
vc_in_name           45
venture_in_name     435
stages              298
partner_in_job     1011
vc_in_job            27
invest_in_job       174
venture_in_job      255
director_in_job     147
dtype: int64

In [114]:
contact_df_tmp["is_vc"] = contact_df_tmp[masks.keys()].sum(axis=1)
contact_df_tmp.head()

,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain,strunz,vc_in_domain,vc_in_name,venture_in_name,partner_in_job,vc_in_job,invest_in_job,venture_in_job,director_in_job,is_vc
hash,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
00003ef5202fac081d992c3f4cd6d0b9,VC Tier 1,,William,Winkenwerder,wwinkenwerder@tigerglobal.com,valid,_nan_,Partner,,Tiger Global Management,tigerglobal.com,"New York, New York, United States",False,,,,,tigerglobal.com,TRUE,TRUE,TRUE,1,tigerglobal.com,False,False,False,False,True,False,False,False,False,1
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
000fcbecf9553942a40ec2ad0f8d88b3,Search4.FinTech_VC_in_US,2023-11-04T04:30:39.811Z,Luiz,Noronha,noronha@dnacapital.com,,https://www.linkedin.com/in/luizhnoronha/,Partner,Helping entrepreneurs build generational compa...,DNA Capital,dnacapital.ca,"Palo Alto, California, United States",False,,,,,dnacapital.com,FALSE,TRUE,FALSE,0,,False,False,False,False,True,False,False,False,False,1
0041bcd7d2bf911652720b947f7e074b,Search4.FinTech_VC_in_US,2023-11-11T14:35:44.417Z,Phil,Herget,ph@ardent.vc,,https://www.linkedin.com/in/phil-herget/,General Partner and Co-Founder,,Ardent Venture Partners,ardent.vc,Washington D.C. Metro Area,False,,,,,ardent.vc,FALSE,TRUE,FALSE,0,,True,True,False,True,True,False,False,False,False,3
00472f7b45ed278dbc1cde93c35fac90,VC_search_export,2024-07-11T18:23:40.118Z,Andrea,Wang,awang@generalcatalyst.com,unknown,https://www.linkedin.com/in/ACwAAAwuMv0BEz4ZcI...,Partner,"Partner to early-stage (pre-seed, seed, series...",General Catalyst,generalcatalyst.com,"San Francisco, California, United States",False,,,,,generalcatalyst.com,FALSE,TRUE,FALSE,0,,False,False,False,False,True,False,False,False,False,1


In [115]:
contact_df_tmp["is_vc"].value_counts()

is_vc
1    788
2    398
0    301
3    240
4     76
5      1
Name: count, dtype: int64

In [116]:
hpandas.filter_df(contact_df_tmp, "is_vc", 0)

INFO  selected=301 / 1804 = 16.69%


,origin,timestamp,first_name,last_name,email,email_verification,linkedin_url,job_title,job_title_description,company_name,company_domain,city,stages,restrictions,industry,category,notes,email_domain,is_ok,both_specified,is_equal,check,actual_domain,strunz,vc_in_domain,vc_in_name,venture_in_name,partner_in_job,vc_in_job,invest_in_job,venture_in_job,director_in_job,is_vc
hash,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0001a9ecb8aa1721852bcd1817e20ad4,GP_LIn_connections,2024-12-31T15:33:29.238Z,Tyrome,Smith,_nan_,,https://linkedin.com/in/tyromesmith,Principal Consultant,Strategic Advisory and Consulting Services | F...,"Go In Now, LLC",go-now.eu,"Bowie, Maryland, United States",False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
00b9de3a32b6af2c2dec58193e94ce9f,VCSheet_Query1,,Mahdi,Raza,mahdi@exponentcap.co,,https://www.linkedin.com/in/mahdiraza92/,Founder@Exponent,,Exponent,exponent.com,,False,,,,,exponentcap.co,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
013c3edff3b4ca24ecc4b4f616efb558,GP_LIn_connections,2024-12-31T02:21:49.506Z,Pete,Wassell,pete@carboneraexchange.com,,https://linkedin.com/in/petewassell,Chairman,"Executive in the field of Software, Technology...",Carbonera Exchange,carboneraexchange.com,New York City Metropolitan Area,False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
01797b1198edb0f3927a7b0aef6f6d6c,GP_LIn_connections,2024-12-31T19:15:56.917Z,Fiona,Smith Foddai,_nan_,,https://linkedin.com/in/fiona-smith-foddai-936...,PATIENT COORDINATOR FOR ASSISTED REPRODUCTION .,Fertility Specialist IVF Coordinator; ;,FIV OCHOA HOSPITAL,_nan_,Greater Marbella Metropolitan Area,False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
02e73ac95db050c2d95585ca3e6ba381,Search4.FinTech_VC_in_US,2023-11-12T14:34:57.731Z,Yubo,Ruan,_nan_,,https://www.linkedin.com/in/yubo-ruan/,Founder,Yubo is a two-time Stanford University dropout...,ParaSpace,para.space,San Francisco Bay Area · Remote,False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
034a8c6009ec436ffc82ee10be8fa40a,VC_search_export,2024-07-11T18:03:20.713Z,Aaron,Kalb,akalb@accel.com,,https://www.linkedin.com/in/ACwAAAOIeb0BdpxrJk...,Entrepreneur in Residence,"Ideating, iterating, and experimenting on the ...",Accel,accel.com,"Palo Alto, California, United States",False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
03e014c6051115125b6464fb55493260,VC_search_export,2024-07-11T18:39:18.011Z,Narendra,Mulani,nmulani@insightpartners.com,accept_all,https://www.linkedin.com/in/ACwAAAAA3isBvH9Pfs...,Senior Advisor,,Insight Partners,insightpartners.com,"New York, New York, United States",False,,,,,insightpartners.com,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
048deeec61c2b1c74118501c28b96825,VCSheet_Query1,,Paul,Graham,pg@ycombinator.com,,https://www.linkedin.com/in/paul-graham-865b4692/,Founder@Y Combinator,,Y Combinator,ycombinator.com,,False,,,,,ycombinator.com,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
04f0721014d8f3d5e9efebdcffeb32d5,Search4.FinTech_VC_in_US,2023-11-13T20:33:23.721Z,Paul,Vassau,_nan_,,https://www.linkedin.com/in/paulvassau/,Chief Revenue Officer (CRO),Advise PE | VC on emerging technologies solvin...,Volcanic Retail,volcanicretail.com,"Provo, Utah, United States · Hybrid",False,,,,,,FALSE,TRUE,FALSE,0,,False,False,False,False,False,False,False,False,False,0
